# Twitter Sentiment Classification: Positive - Neutral - Negative

In [1]:
import pandas as pd

df_train = pd.read_csv('twitter_sentiment_train.csv')
df_test  = pd.read_csv('twitter_sentiment_test.csv')

In [2]:
int_to_label = {2: 'Positive', 1: 'Neutral', 0: 'Negative'}

In [3]:
df_train.head(2)

,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1


In [4]:
y_train=df_train.pop('label')
y_test=df_test.pop('label')

#### Preprocessing Pipeline

In [5]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/neofytos/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/neofytos/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/neofytos/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
import re, string
from ekphrasis.classes.tokenizer import SocialTokenizer
tokenizer = SocialTokenizer(lowercase=False).tokenize

def uncontract(text):
    text = re.sub(r"(\b)([Aa]re|[Cc]ould|[Dd]id|[Dd]oes|[Dd]o|[Hh]ad|[Hh]as|[Hh]ave|[Ii]s|[Mm]ight|[Mm]ust|[Ss]hould|[Ww]ere|[Ww]ould)n't", r"\1\2 not", text)
    text = re.sub(r"(\b)([Hh]e|[Ii]|[Ss]he|[Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Yy]ou)'ll", r"\1\2 will", text)
    text = re.sub(r"(\b)([Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Yy]ou)'re", r"\1\2 are", text)
    text = re.sub(r"(\b)([Ii]|[Ss]hould|[Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Ww]ould|[Yy]ou)'ve", r"\1\2 have", text)

    text = re.sub(r"(\b)([Cc]a)n't", r"\1\2n not", text)
    text = re.sub(r"(\b)([Ii])'m", r"\1\2 am", text)
    text = re.sub(r"(\b)([Ll]et)'s", r"\1\2 us", text)
    text = re.sub(r"(\b)([Ii]t)'s", r"\1\2 is", text)
    text = re.sub(r"(\b)([Tt]here)'s", r"\1\2 is", text)
    text = re.sub(r"(\b)([Ww])on't", r"\1\2ill not", text)
    text = re.sub(r"(\b)([Ss])han't", r"\1\2hall not", text)
    text = re.sub(r"(\b)([Yy])(?:'all|a'll)", r"\1\2ou all", text)

    return text


def tokenize_text(text):
    tokens = tokenizer(text)

    return tokens


def lowercase_tokens(tokens):
    tokens = [t.lower() for t in tokens]

    return tokens


def remove_stopwords(tokens):
    stop_words = stopwords.words('english')

    tokens = [t for t in tokens if t not in stop_words]

    return tokens


def lemmatize(tokens):
    lemmatizer = WordNetLemmatizer()

    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]

    return lemmatized
 
def preprocessing_text(text):
   text = uncontract(text)
   text = tokenize_text(text)
   text = lowercase_tokens(text)
   text = remove_stopwords(text)
   text = lemmatize(text)

   return text

/home/neofytos/nlp/.venv/lib/python3.12/site-packages/ekphrasis/classes/tokenizer.py:225: FutureWarning: Possible nested set at position 2190
  self.tok = re.compile(r"({})".format("|".join(pipeline)))


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Preprocess
clean_train_tweet_tokens=[tokenize_text(text) for text in df_train['text']]
clean_train_tweet_tokens=[lowercase_tokens(tokens) for tokens in clean_train_tweet_tokens]
clean_train_tweet_tokens=[remove_stopwords(tokens) for tokens in clean_train_tweet_tokens]
clean_train_tweet_tokens=[lemmatize(tokens) for tokens in clean_train_tweet_tokens]

clean_tweet_text_train = [' '.join(tokens) for tokens in clean_train_tweet_tokens]

# Train TF-IDF model on preprocessed training data
tfidf_vectorizer = TfidfVectorizer(
   ngram_range  = (1,2),
   max_features = 1000,
   lowercase    = False,
   tokenizer    = None,
   preprocessor = None,
   stop_words   = None,
   min_df       = 10,
   max_df       = 0.80
)
tfidf_vectorizer.fit_transform(clean_tweet_text_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 337360 stored elements and shape (45615, 1000)>

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

def get_tfidf_matrix(texts: list[str], vectorizer: CountVectorizer) -> dict:
    """
    Takes a text string and a fitted CountVectorizer,
    and returns a dictionary with all n-grams in the vectorizer’s vocabulary as keys
    and their corresponding frequencies (counts) in the input text as values.

    Example:
    ----------
    Suppose your fitted vectorizer learned the following vocabulary:
        ["win", "best", "birthday gift", "please call"]

    Input text:
        "Please call now to win a gift! Please call again!"

    Returned dictionary:
        {
          "win": 1,
          "best": 0,
          "birthday gift": 0,
          "please call": 2
        }

    → Words or bigrams not found in the text have a count of 0.
    """
    # Apply preprocessing to text
    texts=[tokenize_text(text) for text in texts]
    texts=[lowercase_tokens(text) for text in texts]
    texts=[remove_stopwords(text) for text in texts]
    texts=[lemmatize(text) for text in texts]

    texts = [' '.join(text) for text in texts]
    
    tfidf_matrix = vectorizer.transform(texts)

    return tfidf_matrix

In [9]:
from pathlib import Path
import joblib
from tqdm import tqdm
from feature_selection import *

# Cache configuration and parameter signature
CACHE_VERSION = "v1"
CACHE_DIR = Path("artifacts") / "cache" / "feature_pipeline"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DATA_PATH = CACHE_DIR / "dataset_features.joblib"
CACHE_VECTORIZER_PATH = CACHE_DIR / "tfidf_vectorizer.joblib"
CACHE_META_PATH = CACHE_DIR / "metadata.joblib"

current_cache_metadata = {
    "cache_version": CACHE_VERSION,
    "tfidf_params": tfidf_vectorizer.get_params(),
    "feature_functions": [
        "log_number_of_tokens",
        "polarity_score",
        "subjectivity_score",
        "exclamation_count",
        "hashtag_count",
        "count_question",
        "mention_count",
        "has_url",
        "vader_scores",
        "count_happy_emoticons",
        "count_sad_emoticons",
        "count_negation",
        "negation_ratio",
        "emoji_sentiment",
        "count_elongated_words",
        "avg_token_length",
        "pos_tag_counts",
        "count_positive_words",
        "count_negative_words",
        "count_profanity"
    ],
    "train_rows": len(df_train),
    "test_rows": len(df_test)
}

def cache_matches_current_params() -> tuple[bool, str]:
    required_files = [CACHE_DATA_PATH, CACHE_VECTORIZER_PATH, CACHE_META_PATH]
    if not all(path.exists() for path in required_files):
        return False, "Missing one or more cache files"

    try:
        saved_metadata = joblib.load(CACHE_META_PATH)
    except Exception as exc:
        return False, f"Could not read cache metadata: {exc}"

    if saved_metadata != current_cache_metadata:
        return False, "Cache metadata differs from current parameters"

    return True, "Cache metadata matches current parameters"

cache_ok, cache_reason = cache_matches_current_params()

if cache_ok:
    cached_data = joblib.load(CACHE_DATA_PATH)
    X_train = cached_data["X_train"]
    X_test = cached_data["X_test"]
    y_train = cached_data["y_train"]
    y_test = cached_data["y_test"]
    tfidf_vectorizer = joblib.load(CACHE_VECTORIZER_PATH)

    # Strict validation to avoid using corrupted or stale cache
    if len(X_train) != len(y_train) or len(X_test) != len(y_test):
        raise ValueError("Cached features and labels have inconsistent lengths")
    if "text" not in X_train.columns or "text" not in X_test.columns:
        raise ValueError("Cached datasets must contain the 'text' column")

    train_feature_cols = [c for c in X_train.columns if c != "text"]
    test_feature_cols = [c for c in X_test.columns if c != "text"]
    if train_feature_cols != test_feature_cols:
        raise ValueError("Feature columns differ between cached train and test sets")

    print(f"Loaded cached features. ({cache_reason})")
else:
    print(f"Cache miss: {cache_reason}. Rebuilding features...")

    X_train = df_train.copy()
    X_test = df_test.copy()

    """
    This block extracts multiple types of text-based features from the dataset (X_train and X_test)
    by applying a list of feature-extraction functions one by one.

    1. The list 'feature_functions' defines all feature extractors to be applied.
       Each function takes an email as input and returns dictionary of features (e.g., POS tag counts, n-gram frequencies).

    2. The for-loop iterates through each feature extraction function with a progress bar (tqdm).

    3. For each function:
          • Apply it to every email in X_train['text'], storing results as a list of dicts.
          • Convert the results to a temporary DataFrame (temp_df).
          • Reset indices and horizontally concatenate (pd.concat) the new features
            to X_train — effectively expanding the dataset with new columns.
          • Repeat the same process for X_test.

    After this loop finishes:
          X_train and X_test contain both the original text column ('text')
          and all derived numeric and categorical features ready for modeling.
    """
    feature_functions = [
    log_number_of_tokens,
    polarity_score,
    subjectivity_score,
    exclamation_count,
    hashtag_count,
    count_question,
    mention_count,
    has_url,
    vader_scores,
    count_happy_emoticons,
    count_sad_emoticons,
    count_negation,
    negation_ratio,
    emoji_sentiment,
    count_elongated_words,
    avg_token_length,
    pos_tag_counts,
    count_positive_words,
    count_negative_words,
    count_profanity
    ]

    for func in tqdm(feature_functions):
        results = X_train['text'].apply(lambda x: func(str(x))).tolist()
        temp_df = pd.DataFrame(results)

        temp_df.reset_index(drop=True, inplace=True)
        X_train.reset_index(drop=True, inplace=True)
        X_train = pd.concat([X_train, temp_df], axis=1)

        results = X_test['text'].apply(lambda x: func(str(x))).tolist()
        temp_df = pd.DataFrame(results)

        temp_df.reset_index(drop=True, inplace=True)
        X_test.reset_index(drop=True, inplace=True)
        X_test = pd.concat([X_test, temp_df], axis=1)

    texts = X_train['text'].astype(str).tolist()

    tfidf_matrix = get_tfidf_matrix(X_train['text'], tfidf_vectorizer)
    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=tfidf_vectorizer.get_feature_names_out()
    )

    X_train = pd.concat([X_train.reset_index(drop=True), tfidf_df], axis=1)

    texts = X_test['text'].astype(str).tolist()

    tfidf_matrix = get_tfidf_matrix(X_test['text'], tfidf_vectorizer)
    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=tfidf_vectorizer.get_feature_names_out()
    )

    X_test = pd.concat([X_test.reset_index(drop=True), tfidf_df], axis=1)

    cache_payload = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }
    joblib.dump(cache_payload, CACHE_DATA_PATH, compress=3)
    joblib.dump(tfidf_vectorizer, CACHE_VECTORIZER_PATH, compress=3)
    joblib.dump(current_cache_metadata, CACHE_META_PATH, compress=3)

    print("Feature cache saved.")

Cache miss: Missing one or more cache files. Rebuilding features...


100%|██████████| 20/20 [14:29<00:00, 43.46s/it] 


Feature cache saved.


In [10]:
X_train.head(10)

,text,log_tokens,polarity,subjectivity,exclamation_count,hashtag_count,question_count,mention_count,has_url,vader_compound,...,yes,yesterday,yet,yoga,york,young,yr,zac,zac brown,zayn
0,"""QT @user In the original draft of the 7th boo...",3.135494,0.375000,0.750000,0,1,0,1,0,0.4588,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"""Ben Smith / Smith (concussion) remains out of...",2.944439,0.000000,0.000000,0,2,0,0,0,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Sorry bout the stream last night I crashed out...,3.258097,0.000000,0.488889,0,0,0,0,0,0.4215,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Chase Headley's RBI double in the 8th inning o...,3.258097,0.000000,0.050000,0,0,0,0,0,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,@user Alciato: Bee will invest 150 million in ...,3.178054,0.000000,0.000000,0,0,0,1,0,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,@user LIT MY MUM 'Kerry the louboutins I wonde...,3.135494,1.000000,0.500000,4,0,0,1,0,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,"""\"""""""" SOUL TRAIN\"""""""" OCT 27 HALLOWEEN SPECIA...",3.806662,0.357143,0.571429,0,0,0,0,0,0.5319,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,So disappointed in wwe summerslam! I want to s...,2.890372,-0.318750,0.475000,1,0,0,0,0,0.2261,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,"""This is the last Sunday w/o football .....,NF...",3.044522,0.000000,0.033333,0,0,0,0,0,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,@user @user CENA & AJ sitting in a tree K-I-S-...,3.737670,0.000000,0.000000,0,0,0,2,0,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
X_train_dropped = X_train.drop(columns=["text"])
X_test_dropped = X_test.drop(columns=["text"])

model = RandomForestClassifier(n_estimators=100, random_state=123)
model.fit(X_train_dropped, y_train)

importances = model.feature_importances_

feature_importances = pd.DataFrame({
    'feature': X_train_dropped.columns,
    'importance': importances
})

sorted_feature_importances=feature_importances.sort_values(by="importance", ascending=False)

topk = sorted_feature_importances.head(50)['feature']

# Top k plot
plt.figure()
plt.barh(topk, sorted_feature_importances[sorted_feature_importances['feature'].isin(topk)]['importance'])
# Highest value at the top
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top k by importance")

KeyError: 'importance'

<Figure size 640x480 with 0 Axes>

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adam
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1775204797.483847   17150 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775204798.184305   17150 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775204800.459069   17150 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [13]:
import numpy as np

model_nn = Sequential([
    Dense(32, input_dim=X_train_dropped.shape[1], activation='relu'),   # hidden layer
    Dense(3, activation='softmax')                              # output layer
])

model_nn.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model_nn.fit(X_train_dropped, y_train, epochs=20, batch_size=32, verbose=1)

# Predict
y_pred_prob_nn = model_nn.predict(X_test_dropped)
y_pred_nn = np.argmax(y_pred_prob_nn, axis=1) 

# Evaluate
print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))

/home/neofytos/nlp/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1775204800.957181   17150 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/20


W0000 00:00:1775204801.634004   17150 cpu_allocator_impl.cc:82] Allocation of 190305780 exceeds 10% of free system memory.


1426/1426 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.5952 - loss: 0.8653
Epoch 2/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6244 - loss: 0.8072
Epoch 3/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6301 - loss: 0.7969
Epoch 4/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6337 - loss: 0.7898
Epoch 5/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6380 - loss: 0.7846
Epoch 6/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6399 - loss: 0.7796
Epoch 7/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6427 - loss: 0.7753
Epoch 8/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6432 - loss: 0.7720
Epoch 9/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6458 - loss: 0.7682
Epoch 10/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6470 - loss: 0.7649
Epoch 11/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6492 - loss: 0.7612
Epoch 12/20
1426/1426 ━━━━━━━━━━━━━━━━━━━

In [14]:
model_nn_topk = Sequential([
    Dense(32, input_dim=X_train_dropped[topk].shape[1], activation='relu'),   # hidden layer
    Dense(3, activation='softmax')                              # output layer
])

model_nn_topk.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model_nn_topk.fit(X_train_dropped[topk], y_train, epochs=20, batch_size=32, verbose=1)

# Predict
y_pred_prob_nn = model_nn_topk.predict(X_test_dropped[topk])
y_pred_nn = np.argmax(y_pred_prob_nn, axis=1) 

# Evaluate
print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))

/home/neofytos/nlp/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20


W0000 00:00:1775204872.370744   17150 cpu_allocator_impl.cc:82] Allocation of 190305780 exceeds 10% of free system memory.


1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.1555 - loss: nan
Epoch 2/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.1555 - loss: nan
Epoch 3/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.1555 - loss: nan
Epoch 4/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.1555 - loss: nan
Epoch 5/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1555 - loss: nan
Epoch 6/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.1555 - loss: nan
Epoch 7/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.1555 - loss: nan
Epoch 8/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.1555 - loss: nan
Epoch 9/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.1555 - loss: nan
Epoch 10/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1555 - loss: nan
Epoch 11/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.1555 - loss: nan
Epoch 12/20
1426/1426 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.1555 